# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [2]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv'
df = pd.read_csv(url)

# Limpiar nombres de columnas
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f"Dataset cargado: {df.shape}")
print(df.columns.tolist())

Dataset cargado: (10910, 26)
['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type']


In [4]:
# ===============================================
# EJERCICIO 1: DataFrame filtrado
# ===============================================

filtered_df = df[
    (df['total_claim_amount'] < 1000) & 
    (df['response'] == 'Yes')
].reset_index(drop=True)

print(f"Total clientes en dataset original: {len(df):,}")
print(f"Clientes filtrados (claim < $1,000 y Response='Yes'): {len(filtered_df):,}")
print(f"Porcentaje del total: {len(filtered_df)/len(df)*100:.1f}%")
print("\nPrimeras filas del DataFrame filtrado:")
filtered_df[['customer', 'response', 'total_claim_amount']].head()

Total clientes en dataset original: 10,910
Clientes filtrados (claim < $1,000 y Response='Yes'): 1,399
Porcentaje del total: 12.8%

Primeras filas del DataFrame filtrado:


,customer,response,total_claim_amount
0,XL78013,Yes,484.013411
1,FM55990,Yes,739.200000
2,CW49887,Yes,547.200000
3,NJ54277,Yes,19.575683
4,MQ68407,Yes,60.036683


In [5]:
# ===============================================
# EJERCICIO 2: Análisis premium, CLV y claims
# ===============================================

yes_df = df[df['response'] == 'Yes']

# Media de premium y CLV para clientes que respondieron "Yes"
avg_premium = yes_df.groupby(['policy_type', 'gender'])['monthly_premium_auto'].mean().round(2)
avg_clv = yes_df.groupby(['policy_type', 'gender'])['customer_lifetime_value'].mean().round(2)

# Media de claims para todos los clientes
avg_claims = df.groupby(['policy_type', 'gender'])['total_claim_amount'].mean().round(2)

# Tabla resumen
tabla2 = pd.DataFrame({
    'avg_monthly_premium_yes': avg_premium,
    'avg_clv_yes': avg_clv,
    'avg_total_claims_all': avg_claims
})

print("ANÁLISIS POR POLICY TYPE Y GENDER")
print("=" * 60)
print(tabla2.to_string())

print("\nCONCLUSIONES:")
print("-" * 60)
print("- El segmento más rentable es el que combina mayor CLV con menor siniestralidad.")
print(f"- Policy type con mayor CLV medio: {avg_clv.idxmax()}")
print(f"- Policy type con menor siniestralidad: {avg_claims.idxmin()}")
print(f"- Policy type con mayor siniestralidad: {avg_claims.idxmax()}")

ANÁLISIS POR POLICY TYPE Y GENDER
                       avg_monthly_premium_yes  avg_clv_yes  avg_total_claims_all
policy_type    gender                                                            
Corporate Auto F                         94.30      7712.63                397.80
               M                         92.19      7944.47                462.22
Personal Auto  F                         99.00      8339.79                413.24
               M                         91.09      7448.38                459.92
Special Auto   F                         92.31      7691.58                458.14
               M                         86.34      8247.09                420.36

CONCLUSIONES:
------------------------------------------------------------
- El segmento más rentable es el que combina mayor CLV con menor siniestralidad.
- Policy type con mayor CLV medio: ('Personal Auto', 'F')
- Policy type con menor siniestralidad: ('Corporate Auto', 'F')
- Policy type con mayor siniest

## Ejercicio 2: Conclusiones

**Segmento más rentable:** Personal Auto – Mujeres
- Mayor CLV medio: $8,339.79
- Premium mensual más alto entre clientes que respondieron "Yes": $99.00
- Siniestralidad media moderada: $413.24

**Segmento más bajo riesgo:** Corporate Auto – Mujeres
- Menor siniestralidad de todos los segmentos: $397.80
- CLV sólido: $7,712.63

**Segmento más alto riesgo:** Corporate Auto – Hombres
- Mayor siniestralidad: $462.22
- CLV más alto del segmento Corporate: $7,944.47 — mayor riesgo pero también mayor valor

**Patrón general:**
- Las mujeres presentan consistentemente menor siniestralidad que los hombres en Corporate y Personal Auto
- Los hombres de Special Auto tienen el CLV más alto ($8,247.09) con siniestralidad moderada — segmento interesante a vigilar
- El premium más bajo (Special Auto – Hombres, $86.34) combinado con CLV alto sugiere una oportunidad de repricing

In [6]:
# ===============================================
# EJERCICIO 3: Clientes por estado (>500)
# ===============================================

state_counts = df.groupby('state')['customer'].count().reset_index()
state_counts.columns = ['state', 'total_customers']
state_counts = state_counts.sort_values('total_customers', ascending=False)

# Filtrar estados con más de 500 clientes
large_states = state_counts[state_counts['total_customers'] > 500]

print("ESTADOS CON MÁS DE 500 CLIENTES")
print("=" * 40)
print(large_states.to_string(index=False))
print(f"\nTotal estados con >500 clientes: {len(large_states)}")
print(f"Total estados en el dataset: {state_counts['state'].nunique()}")

ESTADOS CON MÁS DE 500 CLIENTES
     state  total_customers
California             3552
    Oregon             2909
   Arizona             1937
    Nevada              993
Washington              888

Total estados con >500 clientes: 5
Total estados en el dataset: 5


## Ejercicio 3: Conclusiones

**Estados con más de 500 clientes:**
- California lidera con 3,552 clientes (32.6% del total)
- Oregon es el segundo mercado más grande con 2,909 clientes
- Arizona ocupa el tercer lugar con 1,937 clientes
- Nevada y Washington completan el top 5 con 993 y 888 clientes respectivamente

**Conclusión:** Los 5 estados del dataset superan el umbral de 500 clientes, lo que indica que la compañía opera de forma significativa en todos ellos. California y Oregon concentran más del 50% de la base de clientes total, lo que los convierte en mercados prioritarios para cualquier estrategia de marketing o retención.

In [7]:
# ===============================================
# EJERCICIO 4: CLV stats por Education y Gender
# ===============================================

clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(
    maximum='max',
    minimum='min',
    median='median'
).round(2)

print("CLV POR NIVEL EDUCATIVO Y GÉNERO")
print("=" * 55)
print(clv_stats.to_string())

print(f"\nCombinación con mayor CLV máximo: {clv_stats['maximum'].idxmax()} — ${clv_stats['maximum'].max():,.2f}")
print(f"Combinación con menor CLV mínimo: {clv_stats['minimum'].idxmin()} — ${clv_stats['minimum'].min():,.2f}")
print(f"Combinación con mayor CLV mediano: {clv_stats['median'].idxmax()} — ${clv_stats['median'].max():,.2f}")

CLV POR NIVEL EDUCATIVO Y GÉNERO
                              maximum  minimum   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

Combinación con mayor CLV máximo: ('High School or Below', 'M') — $83,325.38
Combinación con menor CLV mínimo: ('Bachelor', 'M') — $1,898.01
Combinación con mayor CLV mediano: ('High School or Below', 'M') — $6,286.73


## Ejercicio 4: Conclusiones

**CLV máximo:** High School or Below – Hombres ($83,325.38)
- El CLV más alto del dataset pertenece a hombres con nivel educativo básico, lo que sugiere la presencia de outliers significativos en este segmento.

**CLV mínimo:** Bachelor – Hombres ($1,898.01)
- Los valores mínimos son similares en todos los segmentos (entre $1,898 y $2,417), lo que indica un suelo de valor bastante consistente.

**CLV mediano más alto:** High School or Below – Hombres ($6,286.73)
- Este segmento no solo tiene el máximo más alto sino también la mediana más alta, lo que confirma que no es solo efecto de outliers.

**Patrones generales:**
- Los clientes con nivel educativo Bachelor presentan los CLV máximos más altos después de High School (F: $73,225 | M: $67,907)
- Los clientes con nivel Doctor tienen los CLV máximos más bajos — posiblemente menos volumen de pólizas o perfiles más conservadores
- Las diferencias de género son moderadas en la mayoría de niveles educativos, excepto en High School donde los hombres superan claramente a las mujeres ($83,325 vs $55,277)

**Conclusión:** El nivel educativo básico (High School or Below) en hombres representa el segmento con mayor CLV tanto en valor máximo como mediano, lo que lo convierte en un segmento prioritario para estrategias de retención.

In [8]:
# ===============================================
# EJERCICIO 5: Políticas vendidas por estado y mes
# ===============================================

# Convertir fecha a datetime y extraer mes
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.month_name()

# Tabla pivot: estados como filas, meses como columnas
pivot_table = df.pivot_table(
    index='state',
    columns='month',
    values='customer',
    aggfunc='count',
    fill_value=0
)

# Ordenar columnas por mes cronológico
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
existing_months = [m for m in month_order if m in pivot_table.columns]
pivot_table = pivot_table[existing_months]

print("POLÍTICAS VENDIDAS POR ESTADO Y MES")
print("=" * 60)
print(pivot_table.to_string())
print(f"\nTotal políticas: {pivot_table.values.sum():,}")

POLÍTICAS VENDIDAS POR ESTADO Y MES
month       January  February
state                        
Arizona        1008       929
California     1918      1634
Nevada          551       442
Oregon         1565      1344
Washington      463       425

Total políticas: 10,279


C:\Users\juanb\AppData\Local\Temp\ipykernel_26196\1914449475.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])


## Ejercicio 5: Conclusiones

**Políticas vendidas por estado y mes:**

| Estado | Enero | Febrero | Total |
|---|---|---|---|
| California | 1,918 | 1,634 | 3,552 |
| Oregon | 1,565 | 1,344 | 2,909 |
| Arizona | 1,008 | 929 | 1,937 |
| Nevada | 551 | 442 | 993 |
| Washington | 463 | 425 | 888 |

**Patrones observados:**
- El dataset cubre únicamente enero y febrero, por lo que el análisis temporal es limitado.
- California es el estado con mayor volumen de políticas en ambos meses, seguido de Oregon.
- Todos los estados muestran una caída en febrero respecto a enero, lo que podría indicar una tendencia estacional o simplemente que febrero tiene menos días.
- Nevada y Washington son los mercados más pequeños con menos de 1,000 políticas combinadas.

**Conclusión:** La concentración de ventas en California y Oregon sugiere que estos dos estados son el núcleo del negocio y deberían recibir mayor atención en campañas de marketing y retención.

In [9]:
# ===============================================
# EJERCICIO 6: Top 3 estados con más políticas
# ===============================================

# Agrupar por estado y mes
state_month = df.groupby(['state', 'month'])['customer'].count().reset_index()
state_month.columns = ['state', 'month', 'policies_sold']

# Identificar top 3 estados por total de políticas
top3_states = df.groupby('state')['customer'].count().nlargest(3).index.tolist()
print(f"Top 3 estados: {top3_states}")

# Filtrar solo top 3 estados
top3_df = state_month[state_month['state'].isin(top3_states)]

# Tabla pivot con top 3
top3_pivot = top3_df.pivot_table(
    index='state',
    columns='month',
    values='policies_sold',
    aggfunc='sum',
    fill_value=0
)

# Ordenar meses cronológicamente
existing_months = [m for m in month_order if m in top3_pivot.columns]
top3_pivot = top3_pivot[existing_months]
top3_pivot['total'] = top3_pivot.sum(axis=1)
top3_pivot = top3_pivot.sort_values('total', ascending=False)

print("\nPOLÍTICAS VENDIDAS POR MES — TOP 3 ESTADOS")
print("=" * 50)
print(top3_pivot.to_string())

Top 3 estados: ['California', 'Oregon', 'Arizona']

POLÍTICAS VENDIDAS POR MES — TOP 3 ESTADOS
month       January  February  total
state                               
California     1918      1634   3552
Oregon         1565      1344   2909
Arizona        1008       929   1937


## Ejercicio 6: Conclusiones

**Top 3 estados con mayor número de políticas vendidas:**

| Estado | Enero | Febrero | Total |
|---|---|---|---|
| California | 1,918 | 1,634 | 3,552 |
| Oregon | 1,565 | 1,344 | 2,909 |
| Arizona | 1,008 | 929 | 1,937 |

**Patrones observados:**
- California lidera en ambos meses con una diferencia significativa respecto al resto
- Los tres estados muestran el mismo patrón: caída de ventas en febrero respecto a enero
- California representa el 42% del total de políticas de los top 3 estados
- Oregon es el segundo mercado más importante con un volumen cercano al 34% del total top 3

**Conclusión:** California, Oregon y Arizona concentran el 82% de todas las políticas del dataset. Cualquier iniciativa de marketing o retención debería priorizar estos tres mercados, especialmente California donde el volumen es casi el doble que Arizona.

In [10]:
# ===============================================
# EJERCICIO 7: Efecto canal de marketing en respuesta
# ===============================================

# Calcular tasa de respuesta "Yes" por canal
response_by_channel = df.groupby('sales_channel')['response'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).round(2).reset_index()
response_by_channel.columns = ['sales_channel', 'response_rate_%']
response_by_channel = response_by_channel.sort_values('response_rate_%', ascending=False)

# Tabla con volumen total y respuestas Yes
channel_detail = df.groupby('sales_channel').agg(
    total_customers=('customer', 'count'),
    yes_responses=('response', lambda x: (x == 'Yes').sum())
).reset_index()
channel_detail['response_rate_%'] = (channel_detail['yes_responses'] / channel_detail['total_customers'] * 100).round(2)
channel_detail = channel_detail.sort_values('response_rate_%', ascending=False)

print("TASA DE RESPUESTA POR CANAL DE MARKETING")
print("=" * 55)
print(channel_detail.to_string(index=False))
print(f"\nCanal más efectivo: {channel_detail.iloc[0]['sales_channel']} ({channel_detail.iloc[0]['response_rate_%']}%)")
print(f"Canal menos efectivo: {channel_detail.iloc[-1]['sales_channel']} ({channel_detail.iloc[-1]['response_rate_%']}%)")

TASA DE RESPUESTA POR CANAL DE MARKETING
sales_channel  total_customers  yes_responses  response_rate_%
        Agent             4121            742            18.01
          Web             1626            177            10.89
       Branch             3022            326            10.79
  Call Center             2141            221            10.32

Canal más efectivo: Agent (18.01%)
Canal menos efectivo: Call Center (10.32%)


## Ejercicio 7: Conclusiones

**Tasa de respuesta por canal de marketing:**

| Canal | Total Clientes | Respuestas Yes | Tasa de Respuesta |
|---|---|---|---|
| Agent | 4,121 | 742 | 18.01% |
| Web | 1,626 | 177 | 10.89% |
| Branch | 3,022 | 326 | 10.79% |
| Call Center | 2,141 | 221 | 10.32% |

**Patrones observados:**
- El canal Agent es significativamente más efectivo que el resto, con una tasa de respuesta de 18.01% — casi el doble que los demás canales
- Web, Branch y Call Center tienen tasas muy similares, entre 10.32% y 10.89%, sin diferencias significativas entre ellos
- A pesar de tener el mayor volumen de clientes (4,121), Agent mantiene la tasa más alta — lo que indica que la calidad de la interacción supera a los canales digitales o automatizados

**Conclusión:** El canal de agentes humanos es claramente el más efectivo para convertir clientes. La interacción personalizada genera casi el doble de respuestas positivas que cualquier otro canal.